## Pengaturan konfigurasi awal

#### 1. Impor modul yang dibutuhkan

In [85]:
import os
import re
import pandas as pd
from openpyxl import load_workbook

#### 2. Membuat fungsi pendukung

In [86]:
# Membaca data dari file excel
def read_data_from_excel(file_path, sheet_name):
    # Baca sheet dengan header baris 1-3
    df = pd.read_excel(file_path, sheet_name=sheet_name, header=[0,1,2])

    # Gabungkan header multi-level jadi single-level
    df.columns = [
        "_".join([str(c).strip() for c in col if "Unnamed" not in str(c)])
            .lower()
            .replace(" ", "_")
        for col in df.columns.values
    ]

    # Hapus double underscore kalau ada
    df.columns = [c.replace("__", "_").strip("_") for c in df.columns]

    # Cek hasil kolom
    print(df.columns.tolist())

    # Tampilkan data
    print(df.head())

    return df

# Melakukan cleaning data pada dataframe
def cleaning_data(df):
    # bikin copy biar df asli nggak berubah
    df_clean = df.copy()

    # ambil semua kolom object kecuali 'kabupaten/kota'
    object_cols = [col for col in df_clean.select_dtypes(include=["object"]).columns if col != "kabupaten/kota" and col != "provinsi"]

    # convert semua kolom object itu jadi numeric
    # kalau ada "-", "" atau value non-numeric lain -> otomatis jadi NaN
    for col in object_cols:
        df_clean[col] = pd.to_numeric(df_clean[col], errors="coerce")

    # cek hasil konversi
    print(df_clean.dtypes)

    return df_clean

# Mendapatkan data pada periode triwulan berjalan
def get_unpaired_columns(columns):
    # pasangan yang valid
    paired_suffixes = {"ril", "rev", "prov", "kab"}

    unpaired = []

    for col in columns:
        # ambil bagian suffix (setelah underscore paling belakang)
        suffix = col.split("_")[-1]

        # cek apakah suffix masuk pasangan
        if suffix not in paired_suffixes:
            unpaired.append(col)

    return unpaired

# Menggabungkan data evaluasi
def concat_evaluasi(list_evaluasi):
    return pd.concat(list_evaluasi, ignore_index=True)    

# Memastikan nama sheet valid
def safe_sheet_name(name):
    return re.sub(r'[:\\/*?\[\]]', "_", str(name))[:31]

# Menulis data ke dalam file excel
def write_to_excel(df, group_col):
    for group_value in df[group_col].unique():
        df_group = df[df[group_col] == group_value]
        kode = str(df_group["kode"].iloc[0])  # ambil kode group_col

        # Nama file sesuai kode group_col
        if group_col == 'provinsi':
            output_file = f"data/output/evaluasi_{kode[:2]}.xlsx"
            sheet_name = safe_sheet_name(kode[:2] + "00")
        elif group_col == 'kabupaten/kota':
            output_file = f"data/output/evaluasi_{kode[:2]}.xlsx"
            sheet_name = safe_sheet_name(kode[:4])
        

        if os.path.exists(output_file):
            # Load workbook lama
            book = load_workbook(output_file)

            if sheet_name in book.sheetnames:
                # Sheet sudah ada → cari baris terakhir
                ws = book[sheet_name]
                startrow = ws.max_row  # tulis setelah baris terakhir

                with pd.ExcelWriter(output_file, engine="openpyxl", mode="a", if_sheet_exists="overlay") as writer:
                    df_group.to_excel(writer, sheet_name=sheet_name, index=False, header=False, startrow=startrow)
                print(f"Data {group_value} ditambahkan ke sheet {sheet_name} di {output_file}")

            else:
                # Sheet belum ada → tambahkan sheet baru
                with pd.ExcelWriter(output_file, engine="openpyxl", mode="a") as writer:
                    df_group.to_excel(writer, sheet_name=sheet_name, index=False)
                print(f"Sheet {sheet_name} ditambahkan ke {output_file}")

        else:
            # File belum ada → buat baru
            with pd.ExcelWriter(output_file, engine="openpyxl", mode="w") as writer:
                df_group.to_excel(writer, sheet_name=sheet_name, index=False)
            print(f"File baru dibuat: {output_file}, dengan sheet {sheet_name}")


#### 3. Menentukan lokasi file yang akan digunakan sebagai input data

In [87]:
file_path = os.path.join("data/input", "Evaluasi.xlsx")

## Evaluasi Diskrepansi Prov vs KabKot

#### 1. Membaca file Evaluasi.xlsx pada sheet Prov vs KabKot

In [88]:
# Membaca data dari file excel (Diskrepansi 2025)
df = read_data_from_excel(file_path, "Diskrepansi 2025")

# Cleaning data
df_clean = cleaning_data(df)
df_clean.info()

['kode', 'provinsi', 'adhb_pdrb_q1-25', 'adhb_pdrb_q2-25', 'adhb_pdrb_q3-25', 'adhb_pdrb_q4-25', 'adhb_pdrb_2025', 'adhb_pkrt_q1-25', 'adhb_pkrt_q2-25', 'adhb_pkrt_q3-25', 'adhb_pkrt_q4-25', 'adhb_pkrt_2025', 'adhb_pklnprt_q1-25', 'adhb_pklnprt_q2-25', 'adhb_pklnprt_q3-25', 'adhb_pklnprt_q4-25', 'adhb_pklnprt_2025', 'adhb_pkp_q1-25', 'adhb_pkp_q2-25', 'adhb_pkp_q3-25', 'adhb_pkp_q4-25', 'adhb_pkp_2025', 'adhb_pmtb_q1-25', 'adhb_pmtb_q2-25', 'adhb_pmtb_q3-25', 'adhb_pmtb_q4-25', 'adhb_pmtb_2025', 'adhb_pi_q1-25', 'adhb_pi_q2-25', 'adhb_pi_q3-25', 'adhb_pi_q4-25', 'adhb_pi_2025', 'adhb_net_ekspor_q1-25', 'adhb_net_ekspor_q2-25', 'adhb_net_ekspor_q3-25', 'adhb_net_ekspor_q4-25', 'adhb_net_ekspor_2025', 'adhk_pdrb_q1-25', 'adhk_pdrb_q2-25', 'adhk_pdrb_q3-25', 'adhk_pdrb_q4-25', 'adhk_pdrb_2025', 'adhk_pkrt_q1-25', 'adhk_pkrt_q2-25', 'adhk_pkrt_q3-25', 'adhk_pkrt_q4-25', 'adhk_pkrt_2025', 'adhk_pklnprt_q1-25', 'adhk_pklnprt_q2-25', 'adhk_pklnprt_q3-25', 'adhk_pklnprt_q4-25', 'adhk_pklnprt_2

#### 2. Evaluasi diskrepansi ADHB dan ADHK

In [89]:
# 1. Ambil kolom yang diawali adhb atau adhk
target_cols = [col for col in df_clean.columns if col.startswith("adhb") or col.startswith("adhk")]

# 2. Buat list untuk menampung hasil diskrepansi
evaluasi_diskrepansi = []

# 3. Loop tiap kolom target
for col in target_cols:
    for idx, val in df_clean[col].items():
        # PDRB
        if "pdrb" in col:
            # ADHB dan ADHK Q1 & Q2 2025
            if any(q in col for q in ["q1", "q2"]):
                if pd.notna(val) and (val < -5 or val > 5):
                    # Pisahkan nama kolom jadi komponen
                    # contoh: adhb_pdrb_q1_25 → ['adhb', 'pdrb', 'q1', '25']
                    parts = col.split("_")
                    
                    periode = parts[-1]  # ambil q1 / q2

                    # tentukan evaluasi berdasarkan isi col
                    if "adhb" in col:
                        evaluasi_text = "Diskrepansi ADHB antara Provinsi dan total Kabupaten/Kota lebih dari (+-5%)"
                    elif "adhk" in col:
                        evaluasi_text = "Diskrepansi ADHK antara Provinsi dan total Kabupaten/Kota lebih dari (+-5%)"

                    record = {
                        "kode": df_clean.at[idx, "kode"],
                        "provinsi": df_clean.at[idx, "provinsi"],
                        "periode": periode,
                        "nilai": val,
                        "kolom": col,  # tambahan biar tau sumbernya
                        "evaluasi": evaluasi_text
                    }
                    evaluasi_diskrepansi.append(record)

            # ADHB dan ADHK Q3 & Q4 2025
            # elif "q3" in col:
            elif any(q in col for q in ["q3", "q4"]):
                if pd.notna(val) and (val < -3 or val > 3):
                    # Pisahkan nama kolom jadi komponen
                    # contoh: adhb_pdrb_q1_25 → ['adhb', 'pdrb', 'q1', '25']
                    parts = col.split("_")
                    
                    periode = parts[-1]  # ambil q1 / q2

                    # tentukan evaluasi berdasarkan isi col
                    if "adhb" in col:
                        evaluasi_text = "Diskrepansi ADHB antara Provinsi dan total Kabupaten/Kota lebih dari (+-3%)"
                    elif "adhk" in col:
                        evaluasi_text = "Diskrepansi ADHK antara Provinsi dan total Kabupaten/Kota lebih dari (+-3%)"

                    record = {
                        "kode": df_clean.at[idx, "kode"],
                        "provinsi": df_clean.at[idx, "provinsi"],
                        "periode": periode,
                        "nilai": val,
                        "kolom": col,  # tambahan biar tau sumbernya
                        "evaluasi": evaluasi_text
                    }
                    evaluasi_diskrepansi.append(record)
            
            # ADHB dan ADHK Tahunan 2025
            elif "Q" not in col:
                if pd.notna(val) and (val < -2 or val > 2):
                    # Pisahkan nama kolom jadi komponen
                    # contoh: adhb_pdrb_2025 → ['adhb', 'pdrb', '2025']
                    parts = col.split("_")
                    
                    periode = parts[-1]  # ambil tahun

                    # tentukan evaluasi berdasarkan isi col
                    if "adhb" in col:
                        evaluasi_text = "Diskrepansi ADHB antara Provinsi dan total Kabupaten/Kota lebih dari (+-2%)"
                    elif "adhk" in col:
                        evaluasi_text = "Diskrepansi ADHK antara Provinsi dan total Kabupaten/Kota lebih dari (+-2%)"

                    record = {
                        "kode": df_clean.at[idx, "kode"],
                        "provinsi": df_clean.at[idx, "provinsi"],
                        "periode": periode,
                        "nilai": val,
                        "kolom": col,  # tambahan biar tau sumbernya
                        "evaluasi": evaluasi_text
                    }
                    evaluasi_diskrepansi.append(record)
                    
        # Komponen
        elif "pdrb" not in col:
            # ADHB dan ADHK Q1, Q2, Q3 dan Q4 2025
            if pd.notna(val) and (val < -5 or val > 5):
                # Pisahkan nama kolom jadi komponen
                # contoh: adhb_pdrb_q1_25 → ['adhb', 'pdrb', 'q1', '25']
                parts = col.split("_")
                
                periode = parts[-1]  # ambil q1 / q2

                # tentukan evaluasi berdasarkan isi col
                if "adhb" in col:
                    evaluasi_text = "Diskrepansi ADHB antara Provinsi dan total Kabupaten/Kota lebih dari (+-5%)"
                elif "adhk" in col:
                    evaluasi_text = "Diskrepansi ADHK antara Provinsi dan total Kabupaten/Kota lebih dari (+-5%)"
                    
                record = {
                    "kode": df_clean.at[idx, "kode"],
                    "provinsi": df_clean.at[idx, "provinsi"],
                    "periode": periode,
                    "nilai": val,
                    "kolom": col,  # tambahan biar tau sumbernya
                    "evaluasi": evaluasi_text
                }
                evaluasi_diskrepansi.append(record)
                
# 4. Ubah jadi DataFrame biar enak lihat
evaluasi_diskrepansi = pd.DataFrame(evaluasi_diskrepansi)
print(evaluasi_diskrepansi)

     kode             provinsi periode       nilai                 kolom  \
0      96         Papua Tengah   q1-25   21.601929       adhb_pdrb_q1-25   
1      96         Papua Tengah   q2-25  293.857949       adhb_pdrb_q2-25   
2      96         Papua Tengah   q3-25  -12.576441       adhb_pdrb_q3-25   
3      96         Papua Tengah   q4-25   22.336741       adhb_pdrb_q4-25   
4      96         Papua Tengah    2025   87.999993        adhb_pdrb_2025   
..    ...                  ...     ...         ...                   ...   
310    51                 Bali    2025   -5.343587  adhk_net_ekspor_2025   
311    52  Nusa Tenggara Barat    2025   18.116900  adhk_net_ekspor_2025   
312    61     Kalimantan Barat    2025  -31.251644  adhk_net_ekspor_2025   
313    73     Sulawesi Selatan    2025   26.176651  adhk_net_ekspor_2025   
314    96         Papua Tengah    2025  135.650938  adhk_net_ekspor_2025   

                                              evaluasi  
0    Diskrepansi ADHB antara P

#### 3. Evaluasi distribusi

In [90]:
# 1. Ambil semua kolom distribusi
dist_cols = [col for col in df_clean.columns if col.startswith("distribusi")]

# 2. Buat base name tanpa suffix `.1`
pairs = {}
for col in dist_cols:
    # base = re.sub(r'\.1$', '', col)  # buang .1 di akhir
    base = col.replace("p-", "").replace("k-", "")
    pairs.setdefault(base, []).append(col)

# 3. Evaluasi distribusi
evaluasi_dist = []

for base, cols in pairs.items():
    if len(cols) == 2:  # hanya kalau ada pasangan asli + .1
        # col0 = [c for c in cols if not c.endswith(".1")][0]
        # col1 = [c for c in cols if c.endswith(".1")][0]
        col0 = [c for c in cols if ("p-") in c][0]
        col1 = [c for c in cols if ("k-") in c][0]

        # Ambil periode (contoh: distribusi_pdrb_q1-25 → q1)
        parts = base.split("_")
        periode = parts[-1]  # ambil q1-25, q2-25, dst

        for idx in df_clean.index:
            v0 = df_clean.at[idx, col0]
            v1 = df_clean.at[idx, col1]

            # Distribusi PDRB & Komponen Q1,Q2,Q3 2025
            if pd.notna(v0) and pd.notna(v1):
                selisih = abs(v0 - v1)
                if selisih > 5:
                    record = {
                        "kode": df_clean.at[idx, "kode"],
                        "provinsi": df_clean.at[idx, "provinsi"],
                        "periode": periode,
                        "nilai": v1,
                        "kolom": col0 + " vs " + col1,
                        "evaluasi": "Diskrepansi Distribusi antara Provinsi dan total Kabupaten/Kota lebih dari (+-5 point)"
                    }
                    evaluasi_dist.append(record)

# 4. Jadi DataFrame
evaluasi_dist = pd.DataFrame(evaluasi_dist)
print(evaluasi_dist)

    kode      provinsi periode      nilai  \
0     96  Papua Tengah   q1-25  21.109450   
1     96  Papua Tengah   q4-25  22.049664   
2     96  Papua Tengah   q4-25  19.188757   
3     96  Papua Tengah   q1-25   5.641621   
4     96  Papua Tengah   q2-25   5.075478   
5     96  Papua Tengah   q3-25  22.686768   
6     96  Papua Tengah   q4-25 -21.268076   
7     96  Papua Tengah    2025   3.450130   
8     96  Papua Tengah   q1-25  42.271328   
9     96  Papua Tengah   q2-25  46.133059   
10    96  Papua Tengah   q3-25  22.674691   
11    96  Papua Tengah   q4-25  73.047275   

                                                kolom  \
0   distribusi_pkrt_p-q1-25 vs distribusi_pkrt_k-q...   
1   distribusi_pkrt_p-q4-25 vs distribusi_pkrt_k-q...   
2   distribusi_pmtb_p-q4-25 vs distribusi_pmtb_k-q...   
3      distribusi_pi_p-q1-25 vs distribusi_pi_k-q1-25   
4      distribusi_pi_p-q2-25 vs distribusi_pi_k-q2-25   
5      distribusi_pi_p-q3-25 vs distribusi_pi_k-q3-25   
6      distribu

#### 4. Evaluasi growth Q-to-Q, Y-on-Y, C-to-C, Implisit Q-to-Q, dan Implisit Y-on-Y 

In [91]:
# 1. Ambil semua kolom Q-to-Q, Y-on-Y, C-to-C, Implisit Q-to-Q, dan Implisit Y-on-Y 
growth_cols = [col for col in df_clean.columns if col.startswith("q-to-q") or col.startswith("y-on-y") or col.startswith("c-to-c") or col.startswith("implisit_q-to-q") or col.startswith("implisit_y-on-y")]

# 2. Buat base name tanpa suffix `.1`
pairs = {}
for col in growth_cols:
    # base = re.sub(r'\.1$', '', col)  # buang .1 di akhir
    base = col.replace("p-", "").replace("k-", "")
    pairs.setdefault(base, []).append(col)

# 3. Evaluasi Q-to-Q, Y-on-Y, C-to-C, Implisit Q-to-Q, dan Implisit Y-on-Y
evaluasi_growth = []

for base, cols in pairs.items():
    if len(cols) == 2:  # hanya kalau ada pasangan asli + .1
        # col0 = [c for c in cols if not c.endswith(".1")][0]
        # col1 = [c for c in cols if c.endswith(".1")][0]
        col0 = [c for c in cols if ("p-") in c][0]
        col1 = [c for c in cols if ("k-") in c][0]

        # Ambil periode (contoh: q-to-q_pdrb_q1-25 → q1-25)
        parts = base.split("_")
        periode = parts[-1]  # ambil q1-25, q2-25, dst

        for idx in df_clean.index:
            v0 = df_clean.at[idx, col0]
            v1 = df_clean.at[idx, col1]
            
            # Growth Q-to-Q
            if 'q-to-q' in col0 and 'implisit' not in col0:
                # PDRB
                if pd.notna(v0) and pd.notna(v1) and 'pdrb' in col0:
                    selisih = abs(v1 - v0)
                    if v0 * v1 < 0:
                        record = {
                            "kode": df_clean.at[idx, "kode"],
                            "provinsi": df_clean.at[idx, "provinsi"],
                            "periode": periode,
                            "nilai": v1,
                            "kolom": col0 + " vs " + col1,
                            "evaluasi": "Nilai antara Provinsi dan total Kabupaten/Kota beda arah"
                        }
                        evaluasi_growth.append(record)
                    elif selisih > 0.5:
                        record = {
                            "kode": df_clean.at[idx, "kode"],
                            "provinsi": df_clean.at[idx, "provinsi"],
                            "periode": periode,
                            "nilai": v1,
                            "kolom": col0 + " vs " + col1,
                            "evaluasi": "Diskrepansi growth Q-to-Q antara Provinsi dan total Kabupaten/Kota lebih dari (+-0,5 point)"
                        }
                        evaluasi_growth.append(record)
                # Komponen
                elif pd.notna(v0) and pd.notna(v1) and 'pdrb' not in col0:
                    selisih = abs(v1 - v0)
                    if v0 * v1 < 0:
                        record = {
                            "kode": df_clean.at[idx, "kode"],
                            "provinsi": df_clean.at[idx, "provinsi"],
                            "periode": periode,
                            "nilai": v1,
                            "kolom": col0 + " vs " + col1,
                            "evaluasi": "Nilai antara Provinsi dan total Kabupaten/Kota beda arah"
                        }
                        evaluasi_growth.append(record)
                    elif selisih > 1:
                        record = {
                            "kode": df_clean.at[idx, "kode"],
                            "provinsi": df_clean.at[idx, "provinsi"],
                            "periode": periode,
                            "nilai": v1,
                            "kolom": col0 + " vs " + col1,
                            "evaluasi": "Diskrepansi growth Q-to-Q antara Provinsi dan total Kabupaten/Kota lebih dari (+-1 point)"
                        }
                        evaluasi_growth.append(record)

            # Growth Y-on-Y
            elif "y-on-y" in col0 and "implisit" not in col0:
                # PDRB
                if pd.notna(v0) and pd.notna(v1) and 'pdrb' in col0:
                    selisih = abs(v1 - v0)
                    # Q1 dan Q2 2025
                    if any(q in col0 for q in ["q1", "q2"]):
                        if v0 * v1 < 0:
                            record = {
                                "kode": df_clean.at[idx, "kode"],
                                "provinsi": df_clean.at[idx, "provinsi"],
                                "periode": periode,
                                "nilai": v1,
                                "kolom": col0 + " vs " + col1,
                                "evaluasi": "Nilai antara Provinsi dan total Kabupaten/Kota beda arah"
                            }
                            evaluasi_growth.append(record)
                        elif selisih > 0.5:
                            print(selisih)
                            record = {
                                "kode": df_clean.at[idx, "kode"],
                                "provinsi": df_clean.at[idx, "provinsi"],
                                "periode": periode,
                                "nilai": v1,
                                "kolom": col0 + " vs " + col1,
                                "evaluasi": "Diskrepansi growth Y-on-Y antara Provinsi dan total Kabupaten/Kota lebih dari (+-0,5 point)"
                            }
                            evaluasi_growth.append(record)
                    # Q3 dan Q4 2025
                    # elif "q3" in col0:
                    if any(q in col0 for q in ["q3", "q4"]):
                        if v0 * v1 < 0:
                            record = {
                                "kode": df_clean.at[idx, "kode"],
                                "provinsi": df_clean.at[idx, "provinsi"],
                                "periode": periode,
                                "nilai": v1,
                                "kolom": col0 + " vs " + col1,
                                "evaluasi": "Nilai antara Provinsi dan total Kabupaten/Kota beda arah"
                            }
                            evaluasi_growth.append(record)
                        elif selisih > 0.01:
                            record = {
                                "kode": df_clean.at[idx, "kode"],
                                "provinsi": df_clean.at[idx, "provinsi"],
                                "periode": periode,
                                "nilai": v1,
                                "kolom": col0 + " vs " + col1,
                                "evaluasi": "Diskrepansi growth Y-on-Y antara Provinsi dan total Kabupaten/Kota lebih dari (+-0,01 point)"
                            }
                            evaluasi_growth.append(record)
                # Komponen
                elif pd.notna(v0) and pd.notna(v1) and 'pdrb' not in col0:
                    selisih = abs(v1 - v0)
                    # Q1-Q4 2025
                    if any(q in col0 for q in ["q1", "q2", "q3", "q4"]):
                        if v0 * v1 < 0:
                            record = {
                                "kode": df_clean.at[idx, "kode"],
                                "provinsi": df_clean.at[idx, "provinsi"],
                                "periode": periode,
                                "nilai": v1,
                                "kolom": col0 + " vs " + col1,
                                "evaluasi": "Nilai antara Provinsi dan total Kabupaten/Kota beda arah"
                            }
                            evaluasi_growth.append(record)
                        elif selisih > 1:
                            record = {
                                "kode": df_clean.at[idx, "kode"],
                                "provinsi": df_clean.at[idx, "provinsi"],
                                "periode": periode,
                                "nilai": v1,
                                "kolom": col0 + " vs " + col1,
                                "evaluasi": "Diskrepansi growth Y-on-Y antara Provinsi dan total Kabupaten/Kota lebih dari (+-1 point)"
                            }
                            evaluasi_growth.append(record)
                    # Q3 2025
                    # elif "q3" in col0:
                    #     if v0 * v1 < 0:
                    #         record = {
                    #             "kode": df_clean.at[idx, "kode"],
                    #             "provinsi": df_clean.at[idx, "provinsi"],
                    #             "periode": periode,
                    #             "nilai": v1,
                    #             "kolom": col0 + " vs " + col1,
                    #             "evaluasi": "Nilai antara Provinsi dan total Kabupaten/Kota beda arah"
                    #         }
                    #         evaluasi_growth.append(record)
                    #     elif selisih > 0.5:
                    #         record = {
                    #             "kode": df_clean.at[idx, "kode"],
                    #             "provinsi": df_clean.at[idx, "provinsi"],
                    #             "periode": periode,
                    #             "nilai": v1,
                    #             "kolom": col0 + " vs " + col1,
                    #             "evaluasi": "Diskrepansi growth Y-on-Y antara Provinsi dan total Kabupaten/Kota lebih dari (+-0,5 point)"
                    #         }
                    #         evaluasi_growth.append(record)
                
            # Growth C-to-C
            elif 'c-to-c' in col0:
                # PDRB
                if pd.notna(v0) and pd.notna(v1) and 'pdrb' in col0:
                    selisih = abs(v1 - v0)
                    # Q1-Q3 2025
                    if any(q in col0 for q in ["q1", "q2", "q3"]):
                        if v0 * v1 < 0:
                            record = {
                                "kode": df_clean.at[idx, "kode"],
                                "provinsi": df_clean.at[idx, "provinsi"],
                                "periode": periode,
                                "nilai": v1,
                                "kolom": col0 + " vs " + col1,
                                "evaluasi": "Nilai antara Provinsi dan total Kabupaten/Kota beda arah"
                            }
                            evaluasi_growth.append(record)
                        elif selisih > 0.5:
                            record = {
                                "kode": df_clean.at[idx, "kode"],
                                "provinsi": df_clean.at[idx, "provinsi"],
                                "periode": periode,
                                "nilai": v1,
                                "kolom": col0 + " vs " + col1,
                                "evaluasi": "Diskrepansi growth C-to-C antara Provinsi dan total Kabupaten/Kota lebih dari (+-0,5 point)"
                            }
                            evaluasi_growth.append(record)
                    # Q4 2025
                    elif any(q in col0 for q in ["q4"]):
                        if v0 * v1 < 0:
                            record = {
                                "kode": df_clean.at[idx, "kode"],
                                "provinsi": df_clean.at[idx, "provinsi"],
                                "periode": periode,
                                "nilai": v1,
                                "kolom": col0 + " vs " + col1,
                                "evaluasi": "Nilai antara Provinsi dan total Kabupaten/Kota beda arah"
                            }
                            evaluasi_growth.append(record)
                        elif selisih > 0.02:
                            record = {
                                "kode": df_clean.at[idx, "kode"],
                                "provinsi": df_clean.at[idx, "provinsi"],
                                "periode": periode,
                                "nilai": v1,
                                "kolom": col0 + " vs " + col1,
                                "evaluasi": "Diskrepansi growth C-to-C antara Provinsi dan total Kabupaten/Kota lebih dari (+-0,02 point)"
                            }
                            evaluasi_growth.append(record)
                # Komponen
                if pd.notna(v0) and pd.notna(v1) and 'pdrb' not in col0:
                    selisih = abs(v1 - v0)
                    if v0 * v1 < 0:
                        record = {
                            "kode": df_clean.at[idx, "kode"],
                            "provinsi": df_clean.at[idx, "provinsi"],
                            "periode": periode,
                            "nilai": v1,
                            "kolom": col0 + " vs " + col1,
                            "evaluasi": "Nilai antara Provinsi dan total Kabupaten/Kota beda arah"
                        }
                        evaluasi_growth.append(record)
                    elif selisih > 1:
                        record = {
                            "kode": df_clean.at[idx, "kode"],
                            "provinsi": df_clean.at[idx, "provinsi"],
                            "periode": periode,
                            "nilai": v1,
                            "kolom": col0 + " vs " + col1,
                            "evaluasi": "Diskrepansi growth C-to-C antara Provinsi dan total Kabupaten/Kota lebih dari (+-1 point)"
                        }
                        evaluasi_growth.append(record)

            # Growth Implisit Q-to-Q 
            elif 'implisit_q-to-q' in col0:
                if pd.notna(v0) and pd.notna(v1):
                    selisih = abs(v1 - v0)
                    if v0 * v1 < 0:
                        record = {
                            "kode": df_clean.at[idx, "kode"],
                            "provinsi": df_clean.at[idx, "provinsi"],
                            "periode": periode,
                            "nilai": v1,
                            "kolom": col0 + " vs " + col1,
                            "evaluasi": "Nilai antara Provinsi dan total Kabupaten/Kota beda arah"
                        }
                        evaluasi_growth.append(record)
                    elif selisih > 0.5:
                        record = {
                            "kode": df_clean.at[idx, "kode"],
                            "provinsi": df_clean.at[idx, "provinsi"],
                            "periode": periode,
                            "nilai": v1,
                            "kolom": col0 + " vs " + col1,
                            "evaluasi": "Diskrepansi Implisit Q-to-Q antara Provinsi dan total Kabupaten/Kota lebih dari (+-0,5 point)"
                        }
                        evaluasi_growth.append(record)

            # Growth Implisit Y-on-Y 
            elif 'implisit_y-on-y' in col0:
                if pd.notna(v0) and pd.notna(v1):
                    selisih = abs(v1 - v0)
                    if v0 * v1 < 0:
                        record = {
                            "kode": df_clean.at[idx, "kode"],
                            "provinsi": df_clean.at[idx, "provinsi"],
                            "periode": periode,
                            "nilai": v1,
                            "kolom": col0 + " vs " + col1,
                            "evaluasi": "Nilai antara Provinsi dan total Kabupaten/Kota beda arah"
                        }
                        evaluasi_growth.append(record)
                    elif selisih > 1:
                        record = {
                            "kode": df_clean.at[idx, "kode"],
                            "provinsi": df_clean.at[idx, "provinsi"],
                            "periode": periode,
                            "nilai": v1,
                            "kolom": col0 + " vs " + col1,
                            "evaluasi": "Diskrepansi Implisit Y-on-Y antara Provinsi dan total Kabupaten/Kota lebih dari (+-1 point)"
                        }
                        evaluasi_growth.append(record)

# 4. Jadi DataFrame
evaluasi_growth = pd.DataFrame(evaluasi_growth)
print(evaluasi_growth)

0.606672028648342
1.6236568028081502
48.93454616655393
1.5043322977952345
0.512448056934864
1.1031923883492594
     kode             provinsi periode     nilai  \
0      11                 Aceh   q1-25 -5.003814   
1      15                Jambi   q1-25 -2.571448   
2      16     Sumatera Selatan   q1-25  1.688960   
3      21       Kepulauan Riau   q1-25 -1.785336   
4      53  Nusa Tenggara Timur   q1-25 -5.546287   
..    ...                  ...     ...       ...   
903    62    Kalimantan Tengah    2025  3.733235   
904    65     Kalimantan Utara    2025  2.549936   
905    92     Papua Barat Daya    2025 -0.115312   
906    96         Papua Tengah    2025  1.888848   
907    97     Papua Pegunungan    2025  2.481504   

                                                 kolom  \
0           q-to-q_pdrb_p-q1-25 vs q-to-q_pdrb_k-q1-25   
1           q-to-q_pdrb_p-q1-25 vs q-to-q_pdrb_k-q1-25   
2           q-to-q_pdrb_p-q1-25 vs q-to-q_pdrb_k-q1-25   
3           q-to-q_pdrb_p-q1-25 

## Menggabungkan seluruh hasil evaluasi

In [92]:
evaluasi = concat_evaluasi([evaluasi_diskrepansi, evaluasi_dist, evaluasi_growth])
print(evaluasi.head())

   kode      provinsi periode       nilai            kolom  \
0    96  Papua Tengah   q1-25   21.601929  adhb_pdrb_q1-25   
1    96  Papua Tengah   q2-25  293.857949  adhb_pdrb_q2-25   
2    96  Papua Tengah   q3-25  -12.576441  adhb_pdrb_q3-25   
3    96  Papua Tengah   q4-25   22.336741  adhb_pdrb_q4-25   
4    96  Papua Tengah    2025   87.999993   adhb_pdrb_2025   

                                            evaluasi  
0  Diskrepansi ADHB antara Provinsi dan total Kab...  
1  Diskrepansi ADHB antara Provinsi dan total Kab...  
2  Diskrepansi ADHB antara Provinsi dan total Kab...  
3  Diskrepansi ADHB antara Provinsi dan total Kab...  
4  Diskrepansi ADHB antara Provinsi dan total Kab...  


## Menuliskan hasil evaluasi kedalam file excel

In [93]:
write_to_excel(evaluasi, "provinsi")

File baru dibuat: data/output/evaluasi_96.xlsx, dengan sheet 9600
File baru dibuat: data/output/evaluasi_92.xlsx, dengan sheet 9200
File baru dibuat: data/output/evaluasi_91.xlsx, dengan sheet 9100
File baru dibuat: data/output/evaluasi_21.xlsx, dengan sheet 2100
File baru dibuat: data/output/evaluasi_61.xlsx, dengan sheet 6100
File baru dibuat: data/output/evaluasi_62.xlsx, dengan sheet 6200
File baru dibuat: data/output/evaluasi_15.xlsx, dengan sheet 1500
File baru dibuat: data/output/evaluasi_34.xlsx, dengan sheet 3400
File baru dibuat: data/output/evaluasi_73.xlsx, dengan sheet 7300
File baru dibuat: data/output/evaluasi_76.xlsx, dengan sheet 7600
File baru dibuat: data/output/evaluasi_18.xlsx, dengan sheet 1800
File baru dibuat: data/output/evaluasi_52.xlsx, dengan sheet 5200
File baru dibuat: data/output/evaluasi_12.xlsx, dengan sheet 1200
File baru dibuat: data/output/evaluasi_82.xlsx, dengan sheet 8200
File baru dibuat: data/output/evaluasi_95.xlsx, dengan sheet 9500
File baru 

## Evaluasi Revisi & Growth

#### 1. Membaca file Evaluasi.xlsx pada sheet Revisi & Growth

In [94]:
# Membaca data dari file excel (Revisi 2025)
df = read_data_from_excel(file_path, "Revisi 2025")

# Cleaning data
df_clean = cleaning_data(df)
df_clean.info()

['kode', 'kabupaten/kota', 'adhb_pdrb_q1-25_ril', 'adhb_pdrb_q1-25_rev', 'adhb_pdrb_q2-25_ril', 'adhb_pdrb_q2-25_rev', 'adhb_pdrb_q3-25_ril', 'adhb_pdrb_q3-25_rev', 'adhb_pdrb_q4-25', 'adhb_pdrb_2025', 'adhb_pkrt_q1-25_ril', 'adhb_pkrt_q1-25_rev', 'adhb_pkrt_q2-25_ril', 'adhb_pkrt_q2-25_rev', 'adhb_pkrt_q3-25_ril', 'adhb_pkrt_q3-25_rev', 'adhb_pkrt_q4-25', 'adhb_pkrt_2025', 'adhb_pklnprt_q1-25_ril', 'adhb_pklnprt_q1-25_rev', 'adhb_pklnprt_q2-25_ril', 'adhb_pklnprt_q2-25_rev', 'adhb_pklnprt_q3-25_ril', 'adhb_pklnprt_q3-25_rev', 'adhb_pklnprt_q4-25', 'adhb_pklnprt_2025', 'adhb_pkp_q1-25_ril', 'adhb_pkp_q1-25_rev', 'adhb_pkp_q2-25_ril', 'adhb_pkp_q2-25_rev', 'adhb_pkp_q3-25_ril', 'adhb_pkp_q3-25_rev', 'adhb_pkp_q4-25', 'adhb_pkp_2025', 'adhb_pmtb_q1-25_ril', 'adhb_pmtb_q1-25_rev', 'adhb_pmtb_q2-25_ril', 'adhb_pmtb_q2-25_rev', 'adhb_pmtb_q3-25_ril', 'adhb_pmtb_q3-25_rev', 'adhb_pmtb_q4-25', 'adhb_pmtb_2025', 'adhb_pi_q1-25_ril', 'adhb_pi_q1-25_rev', 'adhb_pi_q2-25_ril', 'adhb_pi_q2-25_rev'

#### 2. Evaluasi revisi ADHB dan ADHK

In [95]:
# 1. Ambil semua kolom ADHB dan ADHK 
revisi_harga_cols = [col for col in df.columns if col.startswith("adhb") or col.startswith("adhk")]

# 2. Buat base name tanpa suffix `ril/rev`
pairs = {}
for col in revisi_harga_cols:
    if "ril" in col or "rev" in col:
        base = "_".join(col.split("_")[:-1])  # buang suffix ril/rev
        pairs.setdefault(base, []).append(col)

# 3. Evaluasi ADHB dan ADHK
evaluasi_revisi_harga = []

for base, cols in pairs.items():
    if len(cols) == 2:  # hanya kalau ada pasangan asli + .1
        col0 = [c for c in cols if c.endswith("_ril")][0]
        col1 = [c for c in cols if c.endswith("_rev")][0]

        # Ambil periode (contoh: q-to-q_pdrb_q1-25 → q1-25)
        parts = base.split("_")
        periode = parts[-1]  # ambil q1-25, q2-25, dst

        for idx in df.index:
            v0 = df.at[idx, col0]
            v1 = df.at[idx, col1]

            if pd.notna(v0) and pd.notna(v1) and v1 != 0:
                selisih = abs((v1 - v0) / v0)
                if v0 * v1 < 0:
                    record = {
                        "kode": df.at[idx, "kode"],
                        "kabupaten/kota": df.at[idx, "kabupaten/kota"],
                        "periode": periode,
                        "nilai": v1,
                        "kolom": col0 + " vs " + col1,
                        "evaluasi": "Nilai revisi dan nilai rilis beda arah"
                    }
                    evaluasi_revisi_harga.append(record)
                elif selisih > 0.3:
                    record = {
                        "kode": df.at[idx, "kode"],
                        "kabupaten/kota": df.at[idx, "kabupaten/kota"],
                        "periode": periode,
                        "nilai": v1,
                        "kolom": col0 + " vs " + col1,
                        "evaluasi": "Nilai revisi lebih dari (+-30%) dari nilai rilis"
                    }
                    evaluasi_revisi_harga.append(record)

# 4. Jadi DataFrame
evaluasi_revisi_harga = pd.DataFrame(evaluasi_revisi_harga)
print(evaluasi_revisi_harga)

     kode             kabupaten/kota periode         nilai  \
0    9601           Kabupaten Mimika   q2-25  1.462976e+08   
1    9602          Kabupaten Dogiyai   q2-25  1.646910e+06   
2    9603           Kabupaten Deiyai   q2-25  1.767092e+06   
3    9604           Kabupaten Nabire   q2-25  1.494118e+07   
4    9605           Kabupaten Paniai   q2-25  5.338656e+06   
..    ...                        ...     ...           ...   
502  7503         Kabupaten Pohuwato   q3-25 -9.348273e+03   
503  7602  Kabupaten Polewali Mandar   q3-25 -1.125415e+04   
504  9103    Kabupaten Teluk Wondama   q3-25  7.071635e+03   
505  9201       Kabupaten Raja Ampat   q3-25 -1.352918e+04   
506  9502     Kabupaten Boven Digoel   q3-25 -5.933262e+04   

                                                 kolom  \
0           adhb_pdrb_q2-25_ril vs adhb_pdrb_q2-25_rev   
1           adhb_pdrb_q2-25_ril vs adhb_pdrb_q2-25_rev   
2           adhb_pdrb_q2-25_ril vs adhb_pdrb_q2-25_rev   
3           adhb_pdrb_q

#### 3. Evaluasi revisi distribusi

In [96]:
# 1. Ambil semua kolom distribusi 
revisi_dist_cols = [col for col in df.columns if col.startswith("distribusi")]

# 2. Buat base name tanpa suffix `ril/rev`
pairs = {}
for col in revisi_dist_cols:
    if "ril" in col or "rev" in col:
        base = "_".join(col.split("_")[:-1])  # buang suffix ril/rev
        pairs.setdefault(base, []).append(col)

# 3. Evaluasi distribusi
evaluasi_revisi_dist = []

for base, cols in pairs.items():
    if len(cols) == 2:  # hanya kalau ada pasangan asli + .1
        col0 = [c for c in cols if c.endswith("_ril")][0]
        col1 = [c for c in cols if c.endswith("_rev")][0]

        # Ambil periode (contoh: q-to-q_pdrb_q1-25 → q1-25)
        parts = base.split("_")
        periode = parts[-1]  # ambil q1-25, q2-25, dst

        for idx in df.index:
            v0 = df.at[idx, col0]
            v1 = df.at[idx, col1]

            if pd.notna(v0) and pd.notna(v1):
                selisih = abs(v1 - v0)
                if v0 * v1 < 0:
                    record = {
                        "kode": df.at[idx, "kode"],
                        "kabupaten/kota": df.at[idx, "kabupaten/kota"],
                        "periode": periode,
                        "nilai": v1,
                        "kolom": col0 + " vs " + col1,
                        "evaluasi": "Nilai revisi dan nilai rilis beda arah"
                    }
                    evaluasi_revisi_dist.append(record)
                elif selisih > 5:
                    record = {
                        "kode": df.at[idx, "kode"],
                        "kabupaten/kota": df.at[idx, "kabupaten/kota"],
                        "periode": periode,
                        "nilai": v1,
                        "kolom": col0 + " vs " + col1,
                        "evaluasi": "Nilai revisi lebih dari (+-5 point) dari nilai rilis"
                    }
                    evaluasi_revisi_dist.append(record)

# 4. Jadi DataFrame
evaluasi_revisi_dist = pd.DataFrame(evaluasi_revisi_dist)
print(evaluasi_revisi_dist)

    kode              kabupaten/kota periode      nilai  \
0   3279                 Kota Banjar   q1-25  11.325734   
1   9102           Kabupaten Kaimana   q1-25  46.879614   
2   9204           Kabupaten Maybrat   q1-25  74.821279   
3   9602           Kabupaten Dogiyai   q1-25  52.021616   
4   9603            Kabupaten Deiyai   q1-25  38.953484   
..   ...                         ...     ...        ...   
77  7412  Kabupaten Konawe Kepulauan   q3-25 -23.670695   
78  9203    Kabupaten Sorong Selatan   q3-25 -34.888306   
79  9420            Kabupaten Keerom   q3-25 -82.729048   
80  9502      Kabupaten Boven Digoel   q3-25  -4.998107   
81  9602           Kabupaten Dogiyai   q3-25  -7.505275   

                                                kolom  \
0   distribusi_pkp_q1-25_ril vs distribusi_pkp_q1-...   
1   distribusi_pkp_q1-25_ril vs distribusi_pkp_q1-...   
2   distribusi_pkp_q1-25_ril vs distribusi_pkp_q1-...   
3   distribusi_pkp_q1-25_ril vs distribusi_pkp_q1-...   
4   di

#### 4. Evaluasi revisi growth Q-to-Q, Y-on-Y, C-to-C, Implisit Q-to-Q, Implisit Y-on-Y, SOG Q-to-Q, dan SOG Y-on-Y 

In [97]:
object_cols = df_clean.select_dtypes(include=["object"]).columns.tolist()

# 1. Ambil semua kolom Q-to-Q, Y-on-Y, C-to-C, Implisit Q-to-Q, Implisit Y-on-Y, SOG Q-to-Q, dan SOG Y-on-Y 
revisi_growth_cols = [col for col in df_clean.columns if col.startswith("q-to-q") or col.startswith("y-on-y") or col.startswith("c-to-c") or col.startswith("implisit_q-to-q") or col.startswith("implisit_y-on-y") or col.startswith("sog_q") or col.startswith("sog_y-on-y")]

# 2. Buat base name tanpa suffix `ril/rev`
pairs = {}
for col in revisi_growth_cols:
    if "ril" in col or "rev" in col:
        base = "_".join(col.split("_")[:-1])  # buang suffix ril/rev
        pairs.setdefault(base, []).append(col)

# 3. Evaluasi growth
evaluasi_revisi_growth = []

for base, cols in pairs.items():
    if len(cols) == 2:  # hanya kalau ada pasangan asli + .1
        col0 = [c for c in cols if c.endswith("_ril")][0]
        col1 = [c for c in cols if c.endswith("_rev")][0]

        # Ambil periode (contoh: q-to-q_pdrb_q1-25 → q1-25)
        parts = base.split("_")
        periode = parts[-1]  # ambil q1-25, q2-25, dst

        for idx in df_clean.index:
            v0 = df_clean.at[idx, col0]
            v1 = df_clean.at[idx, col1]

            # Growth Q-to-Q, Y-on-Y, C-to-C, Implisit Q-to-Q, dan Implisit Y-on-Y
            if ('sog' not in col0 and any(x in col0 for x in ['q-to-q', 'y-on-y', 'c-to-c'])):
                # PDRB
                if pd.notna(v0) and pd.notna(v1) and 'pdrb' in col0:
                    if type(v0) == str or type(v1) == str:
                        print("ini nilai v0:", v0, "field:", col0, "Type datanya:", type(v0))
                        print("ini nilai v1:", v1, "field:", col1, "Type datanya:", type(v1)) 

                    selisih = abs(v1 - v0)
                    if v0 * v1 < 0:
                        record = {
                            "kode": df_clean.at[idx, "kode"],
                            "kabupaten/kota": df_clean.at[idx, "kabupaten/kota"],
                            "periode": periode,
                            "nilai": v1,
                            "kolom": col0 + " vs " + col1,
                            "evaluasi": "Nilai revisi dan nilai rilis beda arah"
                        }
                        evaluasi_revisi_growth.append(record)
                    elif selisih > 3:
                        record = {
                            "kode": df_clean.at[idx, "kode"],
                            "kabupaten/kota": df_clean.at[idx, "kabupaten/kota"],
                            "periode": periode,
                            "nilai": v1,
                            "kolom": col0 + " vs " + col1,
                            "evaluasi": "Nilai revisi lebih dari (+-3 point) dari nilai rilis"
                        }
                        evaluasi_revisi_growth.append(record)
                # Komponen
                if pd.notna(v0) and pd.notna(v1) and 'pdrb' not in col0:
                    if type(v0) == str or type(v1) == str:
                        print("ini nilai v0:", v0, "field:", col0, "Type datanya:", type(v0))
                        print("ini nilai v1:", v1, "field:", col1, "Type datanya:", type(v1)) 

                    selisih = abs(v1 - v0)
                    if v0 * v1 < 0:
                        record = {
                            "kode": df_clean.at[idx, "kode"],
                            "kabupaten/kota": df_clean.at[idx, "kabupaten/kota"],
                            "periode": periode,
                            "nilai": v1,
                            "kolom": col0 + " vs " + col1,
                            "evaluasi": "Nilai revisi dan nilai rilis beda arah"
                        }
                        evaluasi_revisi_growth.append(record)
                    elif selisih > 4:
                        record = {
                            "kode": df_clean.at[idx, "kode"],
                            "kabupaten/kota": df_clean.at[idx, "kabupaten/kota"],
                            "periode": periode,
                            "nilai": v1,
                            "kolom": col0 + " vs " + col1,
                            "evaluasi": "Nilai revisi lebih dari (+-4 point) dari nilai rilis"
                        }
                        evaluasi_revisi_growth.append(record)

            # SoG PI (Q-to-Q, Y-on-Y)
            elif any(x in col0 for x in ['sog_q', 'sog_y-on-y']):
                if pd.notna(v0) and pd.notna(v1):
                    if type(v0) == str or type(v1) == str:
                        print("ini nilai v0:", v0, "field:", col0, "Type datanya:", type(v0))
                        print("ini nilai v1:", v1, "field:", col1, "Type datanya:", type(v1)) 

                    if v0 * v1 < 0:
                        record = {
                            "kode": df_clean.at[idx, "kode"],
                            "kabupaten/kota": df_clean.at[idx, "kabupaten/kota"],
                            "periode": periode,
                            "nilai": v1,
                            "kolom": col0 + " vs " + col1,
                            "evaluasi": "Nilai revisi dan nilai rilis beda arah"
                        }
                        evaluasi_revisi_growth.append(record)
                    elif 'pi' in col0:
                        if v1 < -2 or v1 > 2:
                            record = {
                                "kode": df_clean.at[idx, "kode"],
                                "kabupaten/kota": df_clean.at[idx, "kabupaten/kota"],
                                "periode": periode,
                                "nilai": v1,
                                "kolom": col0 + " vs " + col1,
                                "evaluasi": "SoG PI lebih dari +-2 persen"
                            }
                            evaluasi_revisi_growth.append(record)

# 4. Jadi DataFrame
evaluasi_revisi_growth = pd.DataFrame(evaluasi_revisi_growth)
print(evaluasi_revisi_growth)

      kode          kabupaten/kota periode     nilai  \
0     2171              Kota Batam   q1-25  0.655719   
1     3301       Kabupaten Cilacap   q1-25 -1.802729   
2     6503   Kabupaten Tana Tidung   q1-25 -8.016564   
3     9204       Kabupaten Maybrat   q1-25 -4.415943   
4     9502  Kabupaten Boven Digoel   q1-25  0.323967   
...    ...                     ...     ...       ...   
1831  9271             Kota Sorong   q3-25  0.390851   
1832  9420        Kabupaten Keerom   q3-25  3.123848   
1833  9471           Kota Jayapura   q3-25  0.073670   
1834  9502  Kabupaten Boven Digoel   q3-25  4.829790   
1835  9701         Kabupaten Nduga   q3-25 -0.319336   

                                                  kolom  \
0        q-to-q_pdrb_q1-25_ril vs q-to-q_pdrb_q1-25_rev   
1        q-to-q_pdrb_q1-25_ril vs q-to-q_pdrb_q1-25_rev   
2        q-to-q_pdrb_q1-25_ril vs q-to-q_pdrb_q1-25_rev   
3        q-to-q_pdrb_q1-25_ril vs q-to-q_pdrb_q1-25_rev   
4        q-to-q_pdrb_q1-25_ril v

#### 5. Evaluasi growth Q-to-Q, Y-on-Y, C-to-C, Implisit Q-to-Q, Implisit Y-on-Y, SOG Q-to-Q, dan SOG Y-on-Y 

In [ ]:
object_cols = df_clean.select_dtypes(include=["object"]).columns.tolist()

# 1. Ambil semua kolom Q-to-Q, Y-on-Y, C-to-C, Implisit Q-to-Q, Implisit Y-on-Y, SOG Q-to-Q, dan SOG Y-on-Y 
value_growth_cols = [col for col in df_clean.columns if col.startswith("q-to-q") or col.startswith("y-on-y") or col.startswith("c-to-c") or col.startswith("implisit_q-to-q") or col.startswith("implisit_y-on-y") or col.startswith("sog_q") or col.startswith("sog_y-on-y")]

# 2. Buat base name tanpa suffix `ril/rev`
pairs = {}
for col in value_growth_cols:
    if "prov" in col or "kab" in col:
        base = "_".join(col.split("_")[:-1])  # buang suffix ril/rev
        pairs.setdefault(base, []).append(col)

# 3. Evaluasi growth prov vs kab dan triwulan berjalan
evaluasi_value_growth = []

for base, cols in pairs.items():
    if len(cols) == 2:  # hanya kalau ada pasangan asli + .1
        col0 = [c for c in cols if c.endswith("_prov")][0]
        col1 = [c for c in cols if c.endswith("_kab")][0]

        # Ambil periode (contoh: q-to-q_pdrb_q1-25 → q1-25)
        parts = base.split("_")
        periode = parts[-1]  # ambil q1-25, q2-25, dst

        for idx in df_clean.index:
            v0 = df_clean.at[idx, col0]
            v1 = df_clean.at[idx, col1]

            # Growth Q-to-Q, Y-on-Y, C-to-C, Implisit Q-to-Q, dan Implisit Y-on-Y
            if ('sog' not in col0 and any(x in col0 for x in ['q-to-q', 'y-on-y', 'c-to-c'])):
                if pd.notna(v0) and pd.notna(v1):
                    if type(v0) == str or type(v1) == str:
                        print("ini nilai v0:", v0, "field:", col0, "Type datanya:", type(v0))
                        print("ini nilai v1:", v1, "field:", col1, "Type datanya:", type(v1)) 

                    selisih = abs(v1 - v0)
                    if v0 * v1 < 0:
                        record = {
                            "kode": df_clean.at[idx, "kode"],
                            "kabupaten/kota": df_clean.at[idx, "kabupaten/kota"],
                            "periode": periode,
                            "nilai": v1,
                            "kolom": col0 + " vs " + col1,
                            "evaluasi": "Beda arah dengan nilai provinsi"
                        }
                        evaluasi_value_growth.append(record)
                    elif selisih > 4:
                        record = {
                            "kode": df_clean.at[idx, "kode"],
                            "kabupaten/kota": df_clean.at[idx, "kabupaten/kota"],
                            "periode": periode,
                            "nilai": v1,
                            "kolom": col0 + " vs " + col1,
                            "evaluasi": "Selisih dengan nilai provinsi cukup jauh (+-4 point)"
                        }
                        evaluasi_value_growth.append(record)

unpaired_columns = get_unpaired_columns(value_growth_cols)
for col in unpaired_columns:
    for idx, val in df_clean[col].items():
        # PI
        if "pi" in col:
            if pd.notna(val) and (val < -2 or val > 2):
                print(col, idx, val)
                record = {
                    "kode": df_clean.at[idx, "kode"],
                    "kabupaten/kota": df_clean.at[idx, "kabupaten/kota"],
                    # "periode": periode,
                    "periode": col.split("_")[-1],
                    "nilai": val,
                    "kolom": col,
                    "evaluasi": "SoG PI lebih dari +-2 persen"
                }
                evaluasi_value_growth.append(record)

# 4. Jadi DataFrame
evaluasi_value_growth = pd.DataFrame(evaluasi_value_growth)
print(evaluasi_value_growth)

sog_q_pi_q4-25 12 -2.3513670082813607
sog_q_pi_q4-25 13 -4.235652751636844
sog_q_pi_q4-25 21 -3.360251599259862
sog_q_pi_q4-25 24 -2.0724414663392157
sog_q_pi_q4-25 31 -2.497488950600987
sog_q_pi_q4-25 32 -2.5226044813964426
sog_q_pi_q4-25 45 -3.2539429972310963
sog_q_pi_q4-25 49 -2.0593994067762673
sog_q_pi_q4-25 51 -2.1399451774876197
sog_q_pi_q4-25 88 4.3022788677211246
sog_q_pi_q4-25 134 -2.417174421388096
sog_q_pi_q4-25 136 -3.2840953495537604
sog_q_pi_q4-25 143 -4.600767759134969
sog_q_pi_q4-25 145 2.633007845801729
sog_q_pi_q4-25 156 3.4293784829221745
sog_q_pi_q4-25 157 3.628688386052978
sog_q_pi_q4-25 162 2.1133398059707043
sog_q_pi_q4-25 198 -15950.972630205908
sog_q_pi_q4-25 203 -2.4857610065128566
sog_q_pi_q4-25 215 -5.113444157294535
sog_q_pi_q4-25 223 -2.4341857031402823
sog_q_pi_q4-25 231 -3.5941861312390397
sog_q_pi_q4-25 232 -2073185.822107816
sog_q_pi_q4-25 237 -2.630943041497784
sog_q_pi_q4-25 238 -3.4680738490963163
sog_q_pi_q4-25 280 -2.5731137652081917
sog_q_pi_q4

## Menggabungkan seluruh hasil evaluasi

In [99]:
evaluasi = concat_evaluasi([evaluasi_revisi_harga, evaluasi_revisi_dist, evaluasi_revisi_growth, evaluasi_value_growth])
print(evaluasi.head())

   kode     kabupaten/kota periode         nilai  \
0  9601   Kabupaten Mimika   q2-25  1.462976e+08   
1  9602  Kabupaten Dogiyai   q2-25  1.646910e+06   
2  9603   Kabupaten Deiyai   q2-25  1.767092e+06   
3  9604   Kabupaten Nabire   q2-25  1.494118e+07   
4  9605   Kabupaten Paniai   q2-25  5.338656e+06   

                                        kolom  \
0  adhb_pdrb_q2-25_ril vs adhb_pdrb_q2-25_rev   
1  adhb_pdrb_q2-25_ril vs adhb_pdrb_q2-25_rev   
2  adhb_pdrb_q2-25_ril vs adhb_pdrb_q2-25_rev   
3  adhb_pdrb_q2-25_ril vs adhb_pdrb_q2-25_rev   
4  adhb_pdrb_q2-25_ril vs adhb_pdrb_q2-25_rev   

                                           evaluasi  
0  Nilai revisi lebih dari (+-30%) dari nilai rilis  
1  Nilai revisi lebih dari (+-30%) dari nilai rilis  
2  Nilai revisi lebih dari (+-30%) dari nilai rilis  
3  Nilai revisi lebih dari (+-30%) dari nilai rilis  
4  Nilai revisi lebih dari (+-30%) dari nilai rilis  


## Menuliskan hasil evaluasi kedalam file excel

In [100]:
write_to_excel(evaluasi, "kabupaten/kota")

Sheet 9601 ditambahkan ke data/output/evaluasi_96.xlsx
Sheet 9602 ditambahkan ke data/output/evaluasi_96.xlsx
Sheet 9603 ditambahkan ke data/output/evaluasi_96.xlsx
Sheet 9604 ditambahkan ke data/output/evaluasi_96.xlsx
Sheet 9605 ditambahkan ke data/output/evaluasi_96.xlsx
Sheet 9606 ditambahkan ke data/output/evaluasi_96.xlsx
Sheet 9607 ditambahkan ke data/output/evaluasi_96.xlsx
Sheet 9608 ditambahkan ke data/output/evaluasi_96.xlsx
Sheet 3279 ditambahkan ke data/output/evaluasi_32.xlsx
Sheet 7310 ditambahkan ke data/output/evaluasi_73.xlsx
Sheet 8207 ditambahkan ke data/output/evaluasi_82.xlsx
Sheet 7412 ditambahkan ke data/output/evaluasi_74.xlsx
Sheet 8205 ditambahkan ke data/output/evaluasi_82.xlsx
Sheet 8206 ditambahkan ke data/output/evaluasi_82.xlsx
Sheet 8208 ditambahkan ke data/output/evaluasi_82.xlsx
Sheet 9502 ditambahkan ke data/output/evaluasi_95.xlsx
Sheet 1105 ditambahkan ke data/output/evaluasi_11.xlsx
Sheet 1110 ditambahkan ke data/output/evaluasi_11.xlsx
Sheet 1501